# Review figures (Olivier) — preview notebook

Each section below is keyed to a specific comment in Olivier's review. Every
figure is saved as a `*_preview.pdf` in `paper/figures/`; **no manuscript file
is modified** — these are for discussion before deciding what goes in.

| § | Reviewer comment | Output |
|---|---|---|
| 1 | "big version of figure 1, showing 2-3 snapshots … and corresponding isentropic analysis" | `fig1_snapshots_preview.pdf` |
| 2 | "Redo figure 3, with x/y = precipitation rate and organization index (PW) … pdf of precip and OI for each cluster" | `fig3_precip_pw_preview.pdf` |
| 3 | "I would like to see the isentropic mass flux (or anomaly) for the cluster centroid" | `cluster_isentropic_preview.pdf` |
| 4 | "show instead (a) upward mass transport, (b) MSE in updraft/downdraft, (c) upward energy transport per cluster" | `fig5_replacement_preview.pdf` |
| 5 | "I would rather show the net mass flux at selected levels, say 1km and 5km" | `fig6_net_levels_preview.pdf` |
| 6 | "Some time series of the cluster regime … select period during which l2 is positive or negative" | `lambda2_timeseries_preview.pdf` |


In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, Normalize
from scipy.ndimage import gaussian_filter1d
from skimage import measure

sys.path.insert(0, '/Users/sophieabramian/Documents/troposwhere/paper')
import workflow_paper as wfp
from workflow_paper import NUM_STATES, state_colors, cluster_cmap, norm_state

# Fig-1 helpers / styling reused from the proven figure_1_test.py
from figure_1_test import (
    load_simulation, DATA_FOLDER_PATH,
    ocean_cmap, _style_axes3d, _style_axes2d, _style_colorbar,
    BG_COLOR, CLOUD_COLOR, GRID_COLOR, TEXT_COLOR, LIGHT_AZ, LIGHT_ALT,
)

OUT_DIR = '/Users/sophieabramian/Documents/troposwhere/paper/figures'
os.makedirs(OUT_DIR, exist_ok=True)
def _save(fig, name):
    p = os.path.join(OUT_DIR, name)
    fig.savefig(p, dpi=300, bbox_inches='tight', transparent=True)
    print('saved ->', p)


## 1 — Figure 1 expanded: instantaneous snapshots

> *"show snapshots of the isentropic analysis (and not simply the time average)
> … a big version of figure 1, showing 2 or 3 different snapshots of the cloud
> distribution and corresponding isentropic analysis"*

The isentropic panel is now **much smaller** than the 3-D cloud panel (per
Sophie's note). `SNAPSHOT_TIDX` are placeholders — pick the snapshots carefully
later. Needs the LaCie drive mounted (3-D cloud field).

In [ ]:
# ── Config ───────────────────────────────────────────────────────────
SIMU = {'velocity': '8', 'temperature': '300', 'bowen_ratio': '1',
        'microphysic': '1', 'split': '5'}
SPLIT_FILE_IDX = 4                 # split_5.nc (squall-line window)
SNAPSHOT_TIDX  = [120, 250, 400]   # PLACEHOLDER — choose carefully later

ISO_VMIN, ISO_VMAX = -25, 25
ISO_LEVELS = np.unique(np.concatenate([
    np.linspace(-25, -18, 6), np.linspace(-10, -5, 5),
    np.linspace(-5, 5, 15),  np.linspace(5, 10, 5),
    np.linspace(10, 25, 6),
]))


In [ ]:
# ── Load the simulation ──────────────────────────────────────────────
simu = load_simulation(SIMU, i=SPLIT_FILE_IDX)
if simu is None:
    raise RuntimeError('Simulation failed to load — is /Volumes/LaCie mounted?')
backup = os.path.join(DATA_FOLDER_PATH, 'saved_simu')
if os.path.exists(backup):
    simu.load(backup_folder_path=backup)

qn       = simu.dataset_3d.QN.values
vsurf    = simu.dataset_3d.TABS[:, 0]
x        = (simu.dataset_3d.x / 1000).values
y        = (simu.dataset_3d.y / 1000).values
z        = (simu.dataset_3d.z / 1000).values
isentrop = simu.dataset_isentropic.RHO_W_sum.values

nz_iso, nmse = isentrop.shape[1], isentrop.shape[2]
z_iso = z[:nz_iso]
mse   = np.linspace(320, 350, nmse)
nt    = qn.shape[0]
SNAPSHOT_TIDX = [t for t in SNAPSHOT_TIDX if 0 <= t < nt]
print(f'split steps: {nt} | snapshots: {SNAPSHOT_TIDX}')


In [ ]:
def draw_cloud(ax3d, i):
    # 3-D non-precipitating-condensate iso-surface + surface temp anomaly.
    _style_axes3d(ax3d)
    y2 = np.swapaxes(np.array(qn[i]), 0, 2)
    verts, faces, _, _ = measure.marching_cubes(y2, level=0.02)
    verts[:, 0] = x[verts[:, 0].astype(int)]
    verts[:, 1] = y[verts[:, 1].astype(int)]
    verts[:, 2] = z[verts[:, 2].astype(int)]
    vs  = np.array(vsurf[i]); ano = np.clip(vs - vs.mean(), -1.0, 0.0)
    XX, YY = np.meshgrid(x, y)
    ax3d.contourf(XX, YY, ano, 40, zdir='z', offset=-0.05,
                  cmap=ocean_cmap, vmin=-1.0, vmax=0.0, alpha=0.95)
    cl = ax3d.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                           color=CLOUD_COLOR, edgecolor='none', linewidth=0,
                           shade=True, antialiased=True,
                           lightsource=LightSource(azdeg=LIGHT_AZ, altdeg=LIGHT_ALT))
    cl.set_alpha(0.95); cl.set_zorder(5)
    ax3d.set_xticks(np.arange(0, x[-1] + 1, 20))
    ax3d.set_yticks(np.arange(0, y[-1] + 1, 20))
    ax3d.set_zticks(np.arange(0, 13, 5))
    ax3d.grid(which='major', linewidth=0.45, color=GRID_COLOR, alpha=0.9)
    ax3d.set_xlabel('X [km]', labelpad=12, color=TEXT_COLOR)
    ax3d.set_ylabel('Y [km]', labelpad=6,  color=TEXT_COLOR)
    ax3d.set_zlabel('Z [km]', color=TEXT_COLOR)
    ax3d.set_zlim(0, 12); ax3d.view_init(elev=22, azim=-120)
    ax3d.set_box_aspect([1, 1, 0.20])


def draw_isentropic(ax_iso, i):
    # Instantaneous isentropic mass-flux field for time index i (compact panel).
    _style_axes2d(ax_iso)
    field = np.clip(isentrop[i], ISO_VMIN, ISO_VMAX)
    SS, ZZ = np.meshgrid(mse, z_iso)
    cf = ax_iso.contourf(SS, ZZ, field, levels=ISO_LEVELS, cmap='RdBu_r',
                         vmin=ISO_VMIN, vmax=ISO_VMAX, extend='both')
    ax_iso.set_ylim(0, 15)
    ax_iso.set_xlabel('MSE [kJ/kg]', fontsize=8)
    ax_iso.set_ylabel('Height [km]', fontsize=8)
    ax_iso.tick_params(labelsize=7)
    ax_iso.grid(True, color=GRID_COLOR, linewidth=0.4, alpha=0.6)
    return cf


In [ ]:
# ── Compose: one row per snapshot; isentropic panel kept small ───────
n = len(SNAPSHOT_TIDX)
fig = plt.figure(figsize=(12, 4.2 * n), facecolor=BG_COLOR)
# 24 cols: 3-D spans 0:17 (large), isentropic 19:22 (small), gap + cbar after.
gs = fig.add_gridspec(nrows=n, ncols=24, wspace=0.0, hspace=0.30)
pl = 'abcdefghij'
for r, ti in enumerate(SNAPSHOT_TIDX):
    ax3d = fig.add_subplot(gs[r, 0:17], projection='3d', computed_zorder=False)
    # Inset the small isentropic panel vertically so it doesn't dominate the row.
    axiso = fig.add_subplot(gs[r, 19:23])
    axiso.set_box_aspect(1.15)
    draw_cloud(ax3d, ti)
    cf = draw_isentropic(axiso, ti)
    ax3d.set_title(f'({pl[2*r]}) Cloud field — snapshot {r+1} (t-index {ti})',
                   color=TEXT_COLOR, fontsize=11, pad=0)
    axiso.set_title(f'({pl[2*r+1]}) Instantaneous\nisentropic mass flux',
                    color=TEXT_COLOR, fontsize=9, style='italic')

sm = plt.cm.ScalarMappable(norm=Normalize(ISO_VMIN, ISO_VMAX), cmap='RdBu_r')
sm.set_array([])
cb = fig.colorbar(sm, ax=fig.axes, orientation='vertical',
                  fraction=0.012, pad=0.02, aspect=55, extend='both')
cb.set_label(r'Isentropic mass flux [$10^{-3}$ kg m$^{2}$ s$^{-1}$ K$^{-1}$]',
             color=TEXT_COLOR, fontsize=9)
cb.set_ticks(np.linspace(ISO_VMIN, ISO_VMAX, 6))
_style_colorbar(cb, orientation='vertical')
_save(fig, 'fig1_snapshots_preview.pdf')
plt.show()


## Shared data — model latents, states, diagnostics

Loaded once for sections 2–6. Mirrors the setup cell of `figures_paper.ipynb`
so clusters/labels are identical to the paper (1-indexed display). Model
inference over ~19.5k steps takes a few minutes.

In [ ]:
loader, input_dim = wfp.load_dataloader()
model             = wfp.load_model(input_dim=input_dim)
_out              = wfp.extract_latents(model, loader)
embeddings = _out['embeddings']
states     = _out['states']
emb_pca, _ = wfp.run_pca(embeddings)
_, keep_idx = wfp.keep_closest_latent_samples(embeddings, states)

mass_flux = np.load(wfp.PATHS['data_raw'])[:-1]          # (N, z=48, mse=48)
z_array   = wfp.load_z_array()                            # metres, len 48
s_array   = np.linspace(320, 350, 48)                     # MSE bins
pw        = np.load(wfp.PATHS['pw'])[:len(states)]        # organization index
prec      = np.load(wfp.PATHS['prec'])[:len(states)]      # precipitation rate

n = min(len(mass_flux), len(embeddings))
mass_flux = mass_flux[:n]
states_mf = states[:n]

Tmat = wfp.compute_transition_matrix(states, NUM_STATES, future=5)
eigvals, eigvecs = wfp.eigendecomposition(Tmat)
print('states:', len(states), '| mass_flux aligned:', mass_flux.shape)
print('|eigvals|:', np.round(np.abs(eigvals), 4))


## 2 — Figure 3 redo: precipitation rate vs organization index

> *"Redo figure 3, with the x and y axis being precipitation rate and
> organization index (PW … a measure of convective organization). Show the pdf
> of precip and OI for each cluster instead of PCA 1 and 2."*

PCA axes replaced by **precipitation rate** (x) and **organization index PW**
(y). Marginals are now per-cluster PDFs of precip (top) and OI (right).

In [ ]:
fig = plt.figure(figsize=(9, 8))
ax_m = plt.axes([0.12, 0.12, 0.65, 0.65])
ax_t = plt.axes([0.12, 0.80, 0.65, 0.15], sharex=ax_m)
ax_r = plt.axes([0.80, 0.12, 0.15, 0.65], sharey=ax_m)

uniq, cnts = np.unique(states, return_counts=True)
cdict = dict(zip(uniq.astype(int), cnts.astype(int)))
for s in range(NUM_STATES):
    m = states == s
    ax_m.scatter(prec[m], pw[m], s=22, alpha=0.45, linewidth=0.2,
                 facecolors=state_colors[s],
                 label=f'State {wfp.state_label(s)} (n={cdict.get(s, 0)})')
    h, b = np.histogram(prec[m], bins=40, density=True)
    cx = (b[:-1] + b[1:]) / 2
    ax_t.fill_between(cx, gaussian_filter1d(h, 1), alpha=0.22, color=state_colors[s])
    ax_t.plot(cx, gaussian_filter1d(h, 1), color=state_colors[s], lw=2)
    h2, b2 = np.histogram(pw[m], bins=40, density=True)
    cy = (b2[:-1] + b2[1:]) / 2
    ax_r.fill_betweenx(cy, gaussian_filter1d(h2, 1), alpha=0.22, color=state_colors[s])
    ax_r.plot(gaussian_filter1d(h2, 1), cy, color=state_colors[s], lw=2)

ax_m.set_xlabel('Precipitation rate'); ax_m.set_ylabel('Organization index (PW)')
ax_t.set_ylabel('Density'); ax_r.set_xlabel('Density')
ax_t.tick_params(axis='x', labelbottom=False)
ax_r.tick_params(axis='y', labelleft=False)
for a in (ax_m, ax_t, ax_r):
    a.grid(True, linestyle='--', alpha=0.35)
ax_m.legend(loc='upper right', fontsize=9, framealpha=0.9)
_save(fig, 'fig3_precip_pw_preview.pdf')
plt.show()


## 3 — Cluster-centroid isentropic mass flux (and anomaly)

> *"I would like to see the isentropic mass flux (or anomaly) for the cluster
> centroid, or at least a few of them (suggestion: clusters 2,3,5,6)."*

Top row: per-cluster **mean** isentropic mass-flux field (z × MSE). Bottom row:
**anomaly** vs the all-time mean. All 7 clusters shown; Olivier's suggested 2,3,5,6
are simply the corresponding columns.

In [ ]:
mf_mean_all = mass_flux.mean(axis=0)
SS, ZZ = np.meshgrid(s_array, z_array / 1000.0)   # MSE × height(km)

fig, axes = plt.subplots(2, NUM_STATES, figsize=(3.0 * NUM_STATES, 7),
                         sharex=True, sharey=True)
vlim_m = np.percentile(np.abs([mass_flux[states_mf == s].mean(0)
                               for s in range(NUM_STATES)]), 99)
vlim_a = np.percentile(np.abs([mass_flux[states_mf == s].mean(0) - mf_mean_all
                               for s in range(NUM_STATES)]), 99)
for s in range(NUM_STATES):
    fld = mass_flux[states_mf == s].mean(0)
    axes[0, s].contourf(SS, ZZ, fld, levels=20, cmap='RdBu_r',
                        vmin=-vlim_m, vmax=vlim_m, extend='both')
    axes[1, s].contourf(SS, ZZ, fld - mf_mean_all, levels=20, cmap='RdBu_r',
                        vmin=-vlim_a, vmax=vlim_a, extend='both')
    axes[0, s].set_title(f'State {wfp.state_label(s)}', color=state_colors[s],
                         fontweight='bold')
    for row in (0, 1):
        axes[row, s].set_ylim(0, 15)
        axes[row, s].grid(True, linestyle='--', alpha=0.3)
axes[0, 0].set_ylabel('mean\nHeight [km]')
axes[1, 0].set_ylabel('anomaly\nHeight [km]')
for s in range(NUM_STATES):
    axes[1, s].set_xlabel('MSE [kJ/kg]')
fig.colorbar(plt.cm.ScalarMappable(Normalize(-vlim_m, vlim_m), 'RdBu_r'),
             ax=axes[0, :], fraction=0.012, pad=0.01, label='mass flux')
fig.colorbar(plt.cm.ScalarMappable(Normalize(-vlim_a, vlim_a), 'RdBu_r'),
             ax=axes[1, :], fraction=0.012, pad=0.01, label='anomaly')
_save(fig, 'cluster_isentropic_preview.pdf')
plt.show()


## 4 — Figure 5 replacement

> *"I would show instead (a) the upward mass transport, (b) the profile of MSE
> in the updraft and downdraft, and (c) the upward energy transport associated
> with each cluster."*

Per-cluster vertical profiles (cluster mean, not anomaly, per Olivier):
- **(a)** upward mass transport = Σ over MSE of the positive mass flux.
- **(b)** mass-flux-weighted mean MSE in the updraft (solid) and downdraft (dashed):
  ⟨MSE⟩↑ = Σ(F⁺·MSE)/Σ F⁺, and likewise with F⁻.
- **(c)** upward energy transport = Σ over MSE of (F⁺ · MSE) (proportional to
  the MSE flux carried upward).

In [ ]:
mf_pos = np.clip(mass_flux, 0, None)
mf_neg = np.clip(mass_flux, None, 0)
zkm = z_array / 1000.0
eps = 1e-12

fig, axes = plt.subplots(1, 3, figsize=(13, 5.5), sharey=True)
for s in range(NUM_STATES):
    idx = keep_idx.get(s)
    idx = None if idx is None else idx[idx < len(mass_flux)]
    if idx is None or len(idx) == 0:
        continue
    fp = mf_pos[idx].mean(0)            # (z, mse)
    fn = mf_neg[idx].mean(0)
    up_mass = fp.sum(axis=1)
    mse_up  = (fp * s_array[None, :]).sum(1) / (fp.sum(1) + eps)
    mse_dn  = (fn * s_array[None, :]).sum(1) / (fn.sum(1) - eps)
    up_ener = (fp * s_array[None, :]).sum(axis=1)
    c = state_colors[s]
    axes[0].plot(up_mass, zkm, color=c, lw=2, label=f'State {wfp.state_label(s)}')
    axes[1].plot(mse_up, zkm, color=c, lw=2)
    axes[1].plot(mse_dn, zkm, color=c, lw=1.4, linestyle='--')
    axes[2].plot(up_ener, zkm, color=c, lw=2)

axes[0].set_title('(a) Upward mass transport')
axes[1].set_title('(b) Mean MSE: updraft (—) / downdraft (- -)')
axes[2].set_title('(c) Upward energy transport')
axes[0].set_ylabel('Height [km]')
for a, xl in zip(axes, ['Σ F⁺', 'MSE [kJ/kg]', 'Σ F⁺·MSE']):
    a.set_xlabel(xl); a.set_ylim(0, 15); a.grid(True, linestyle='--', alpha=0.3)
axes[0].legend(fontsize=8, ncol=2)
fig.tight_layout()
_save(fig, 'fig5_replacement_preview.pdf')
plt.show()


## 5 — Figure 6 redo: up & down mass flux at selected levels

> *"I find the vertical integral of the mass flux less useful. I would rather
> show the net mass flux at selected levels, say 1km and 5km."*

Plotting **gross up (F⁺)** and **gross down (F⁻)** *separately*, not the net.
Rationale: in this dataset the per-level net carries a known binning artifact
(non-anomaly `RHO_W` + truncated FMSE window), so the *net* at any single
level is biased. The up and down branches are the physically meaningful
quantities and per-cluster differences in them are what one actually wants to
compare. Two panels: ~1 km and ~5 km. Per cluster, an upward bar (gross up)
and a downward bar (gross down) summed over MSE.

In [ ]:
z1 = int(np.argmin(np.abs(z_array - 1000.)))
z5 = int(np.argmin(np.abs(z_array - 5000.)))
print(f'nearest levels: z[{z1}]={z_array[z1]:.0f} m, z[{z5}]={z_array[z5]:.0f} m')

up_z = np.clip(mass_flux, 0, None).sum(axis=2)   # (N, z) gross upward
dn_z = np.clip(mass_flux, None, 0).sum(axis=2)   # (N, z) gross downward

lab = [wfp.state_label(s) for s in range(NUM_STATES)]
cols = [state_colors[s] for s in range(NUM_STATES)]

def _per_state(arr_z, zi):
    return [arr_z[states_mf == s, zi].mean() for s in range(NUM_STATES)]

up1, dn1 = _per_state(up_z, z1), _per_state(dn_z, z1)
up5, dn5 = _per_state(up_z, z5), _per_state(dn_z, z5)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
xpos = np.arange(NUM_STATES)
for ax, (up_v, dn_v, zlab) in zip(
        axes,
        [(up1, dn1, f'~1 km ({z_array[z1]:.0f} m)'),
         (up5, dn5, f'~5 km ({z_array[z5]:.0f} m)')]):
    ax.bar(xpos, up_v, color=cols, edgecolor='k', alpha=0.95, label='gross up (F⁺)')
    ax.bar(xpos, dn_v, color=cols, edgecolor='k', alpha=0.55, hatch='//',
           label='gross down (F⁻)')
    ax.axhline(0, color='k', lw=1)
    ax.set_xticks(xpos); ax.set_xticklabels(lab)
    ax.set_xlabel('State / Cluster')
    ax.set_title(f'Isentropic mass flux at {zlab}')
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Mass flux (Σ over MSE)')
axes[0].legend(loc='upper right', fontsize=9)
fig.tight_layout()
_save(fig, 'fig6_net_levels_preview.pdf')
plt.show()


## 6 — λ₂-conditioned regime time series

> *"Some time series of the cluster regime and transition would be quite useful
> … I would select periods during which l2 is either positive or negative."*

The second eigenvector of the transition matrix gives a per-state loading on the
slow mode λ₂. We build a λ₂ score time series, `v2[state_t]`, smooth it, then
show one window where it is persistently **positive** and one where it is
persistently **negative** — with the state sequence colour-coded by cluster.

In [ ]:
v2 = np.real(eigvecs[:, 1])
v2 = v2 / np.max(np.abs(v2))
order = np.argsort(v2)
print('lambda2 =', np.round(np.real(eigvals[1]), 4),
      '| per-state v2:', dict(zip([wfp.state_label(s) for s in range(NUM_STATES)],
                                  np.round(v2, 2))))

score = gaussian_filter1d(v2[states].astype(float), 25)

def _window(positive, length=600):
    sgn = score > 0 if positive else score < 0
    best, run, start = (0, 0), 0, 0
    for i, b in enumerate(sgn):
        if b:
            run = run + 1 if run else 1
            start = i - run + 1 if run == 1 else start
            if run > best[1]:
                best = (start, run)
        else:
            run = 0
    s0 = max(0, best[0] + best[1] // 2 - length // 2)
    return s0, min(len(states), s0 + length)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
for ax, pos, ttl in zip(axes, (True, False),
                        ('λ₂ persistently positive', 'λ₂ persistently negative')):
    a, b = _window(pos)
    t = np.arange(a, b)
    ax.scatter(t, states[a:b] + 1, c=[state_colors[s] for s in states[a:b]],
               s=14, edgecolor='none')
    ax2 = ax.twinx()
    ax2.plot(t, score[a:b], color='k', lw=1.5, alpha=0.7)
    ax2.axhline(0, color='grey', lw=0.8, linestyle='--')
    ax2.set_ylabel('λ₂ score', color='k')
    ax.set_yticks(range(1, NUM_STATES + 1))
    ax.set_ylabel('State / Cluster'); ax.set_title(f'{ttl}  (steps {a}-{b})')
    ax.grid(True, axis='x', linestyle='--', alpha=0.3)
axes[-1].set_xlabel('Time step')
fig.tight_layout()
_save(fig, 'lambda2_timeseries_preview.pdf')
plt.show()
